In [ ]:
import sys
import os
from pathlib import Path  # noqa: F401

# Resolve project root regardless of where the notebook is launched from
for _candidate in [".", "..", "../.."]:
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

import seaborn as sns  # noqa: E402

from src.analysis.isc import (  # noqa: E402
    FREQUENCY_BANDS,
    compute_loo_isc,
    compute_sliding_window_isc,
)
from src.definitions.constants import ProjectPaths  # noqa: E402
from src.definitions.fields import (  # noqa: E402
    ConditionVariants,
    ExclusionCategories,
    MusicTypeVariants,
)
from src.visualization.isc_plots import (  # noqa: E402
    plot_band_isc_distributions,
    plot_band_mean_isc_bar,
    plot_band_overlap,
    plot_band_sliding_window_isc,
    print_band_significant_intervals,
    print_data_overview,
)
from scripts.analysis_common import (  # noqa: E402
    BAND_ISC_THRESHOLDS,
    analyzers_to_datasets,
    load_analyzers,
)

%matplotlib inline
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.0)
print("Setup complete.")

# EEG Inter-Subject Correlation Analysis — Per Frequency Band

This notebook computes **per-band inter-subject correlation (ISC)** on the raw EEG
signal band-pass filtered to the five standard EEG frequency bands
(delta, theta, alpha, beta, gamma):

1. **Per-band LOO-ISC distributions** — histogram of mean channel-wise LOO-ISC per
   band, plus a summary bar chart of mean feature-averaged ISC per band.
2. **Per-band sliding-window ISC** — time-resolved LOO-ISC for each frequency band
   with per-channel heatmap and significant interval printout.
3. **Band-overlap analysis** — raster plot showing which time windows are
   simultaneously significant in multiple bands and/or broadband.

All computation uses `src.analysis.isc` and all visualisation uses
`src.visualization.isc_plots`.  Data are loaded **without normalization**.

> **Production script:** `scripts/run_isc.py`  
> **Broadband analysis:** `isc_broadband.ipynb`

## Configuration

In [ ]:
# ── Experiment configuration ─────────────────────────────────────────────────
CONDITION = ConditionVariants.PLACEBO
MUSIC_TYPES = [MusicTypeVariants.CLASSICAL, MusicTypeVariants.PSYTRANCE]
EXCLUSION_CATEGORIES = [ExclusionCategories.BAD_MUSIC, ExclusionCategories.ARTIFACTS]

# ── Sliding-window ISC parameters ────────────────────────────────────────────
WINDOW_SEC = 5.0  # sliding-window length in seconds
STEP_SEC = 2.5  # sliding-window step size in seconds
ISC_THRESHOLD = 0.035  # broadband significance threshold (Pearson r)

# ── Data processing flag ──────────────────────────────────────────────────────
# Set True to load raw .fif files, resample, stack, and save before analysis.
# Keep False to use already-saved concatenated arrays.
process_and_save_data = False

# ── Plot saving ──────────────────────────────────────────────────────────────
# Set SAVE_PLOTS=False to only display figures inline without saving.
SAVE_PLOTS = True
PLOTS_DIR = ProjectPaths.NOTEBOOKS_DIR / "02-isc-broadband-analysis" / "plots" / "bands"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Plots will be saved to: {PLOTS_DIR}")
print(f"Frequency bands : {list(FREQUENCY_BANDS.keys())}")
print(f"Band ISC thresholds: {BAND_ISC_THRESHOLDS}")

## Data Loading

In [ ]:
# Load (or process-and-save) one EEGSummarizedAnalyzer per music type.
# normalize_data=False: keep raw amplitudes for this ISC analysis.
analyzers = load_analyzers(
    MUSIC_TYPES,
    CONDITION,
    EXCLUSION_CATEGORIES,
    process_and_save_data,
    normalize_data=False,
)
datasets = analyzers_to_datasets(analyzers)
print_data_overview(datasets)

## Dataset Selection

Change `LABEL` to switch between music types.  All analysis cells below use
`ad`, `data`, `n_subjects`, `n_channels`, `n_times`, and `sfreq`.

In [ ]:
LABEL = MusicTypeVariants.CLASSICAL.value
# LABEL = MusicTypeVariants.PSYTRANCE.value

ad = datasets[LABEL]
data = ad.data  # (n_subjects, n_channels, n_times) — unnormalized
sfreq = ad.sfreq

n_subjects, n_channels, n_times = data.shape

print(f"Dataset : {LABEL}")
print(f"Shape   : {data.shape}  (subjects × channels × time points)")
print(f"Duration: {n_times / sfreq:.1f} s  @  {sfreq} Hz")
print("Data: unnormalized (raw amplitude).")

## Section 1 — Per-Band LOO-ISC

The raw EEG data is band-pass filtered to each standard frequency band and
leave-one-out ISC is computed on the filtered signal.

- `band_iscs` dict: `{band_name: (loo_isc, mean_loo_isc)}` where
  `loo_isc` shape is `(n_subjects, n_channels)` and
  `mean_loo_isc` shape is `(n_channels,)`.
- **Histogram subplot** per band — distribution of `mean_loo_isc` across channels.
- **Bar chart** — mean feature-averaged LOO-ISC ± SD across all bands.

In [ ]:
band_iscs: dict[str, tuple] = {}
for band, (l_freq, h_freq) in FREQUENCY_BANDS.items():
    filtered = ad.filter_to_band(l_freq, h_freq)
    loo, mean_isc = compute_loo_isc(filtered.data)
    band_iscs[band] = (loo, mean_isc)
    print(f"  {band:6s}  loo_isc={loo.shape}  mean={mean_isc.mean():.4f}")

fig_dist = plot_band_isc_distributions(
    {LABEL: band_iscs},
    bands=FREQUENCY_BANDS,
    feature_axis_label="Number of channels",
    save_path=PLOTS_DIR / "band_isc_distributions.png" if SAVE_PLOTS else None,
)

fig_bar = plot_band_mean_isc_bar(
    {LABEL: band_iscs},
    bands=FREQUENCY_BANDS,
    save_path=PLOTS_DIR / "band_isc_mean_bar.png" if SAVE_PLOTS else None,
)

## Section 2 — Per-Band Sliding-Window ISC

LOO-ISC is computed in overlapping windows of `WINDOW_SEC` seconds (step `STEP_SEC`
seconds) on each band-filtered signal.

- `band_sw` dict: `{band_name: (isc_timecourse, window_times)}`
- Each subplot shows mean LOO-ISC time course + per-channel heatmap for one band.
- Per-band significance thresholds from `BAND_ISC_THRESHOLDS` are used for shading
  and for the significant interval printout.

In [ ]:
band_sw: dict[str, tuple] = {}
for band, (l_freq, h_freq) in FREQUENCY_BANDS.items():
    filtered = ad.filter_to_band(l_freq, h_freq)
    tc, times = compute_sliding_window_isc(
        filtered.data,
        window_sec=WINDOW_SEC,
        step_sec=STEP_SEC,
        sfreq=sfreq,
    )
    band_sw[band] = (tc, times)
    print(f"  {band:6s}  isc_tc={tc.shape}  times={times.shape}")

fig_sw = plot_band_sliding_window_isc(
    {LABEL: band_sw},
    bands=FREQUENCY_BANDS,
    isc_threshold=BAND_ISC_THRESHOLDS,
    feature_axis_label="Channel",
    save_path=PLOTS_DIR / "band_sliding_window_isc.png" if SAVE_PLOTS else None,
)

print_band_significant_intervals(
    {LABEL: band_sw},
    bands=FREQUENCY_BANDS,
    band_thresholds=BAND_ISC_THRESHOLDS,
    default_threshold=ISC_THRESHOLD,
)

## Section 3 — Band-Overlap Analysis

A raster plot showing, for each time window, which frequency bands (and the
broadband signal) simultaneously exceed their respective ISC significance thresholds.

The broadband sliding-window ISC is computed here using the same window parameters
as the per-band analysis.

In [ ]:
# Broadband sliding-window ISC (required for the overlap raster)
sw_isc, sw_times = compute_sliding_window_isc(
    data,
    window_sec=WINDOW_SEC,
    step_sec=STEP_SEC,
    sfreq=sfreq,
)

fig_overlap = plot_band_overlap(
    {LABEL: band_sw},
    bands=FREQUENCY_BANDS,
    band_thresholds=BAND_ISC_THRESHOLDS,
    broadband_sw={LABEL: (sw_isc, sw_times)},
    broadband_threshold=ISC_THRESHOLD,
    save_path=PLOTS_DIR / "band_overlap.png" if SAVE_PLOTS else None,
)